# SPORA CODEX: basic statistics and visualization
2026-11-11 · LLY

Explore **haist2026lymph**, **hickey2023organization**, and **hoefflin2026spatial** in the local REQ-82797 download. This notebook summarizes tissue/patient counts, modality pairing, splits, channel coverage, image dimensions, metadata quality, and available cell annotations. Image previews use bounded crops; source files are read only.

Run cells from top to bottom with Python 3.11+ and `numpy pandas pyarrow matplotlib zarr>=3 nbformat ipykernel`. If needed, install these into your notebook environment with `%pip install numpy pandas pyarrow matplotlib "zarr>=3" nbformat ipykernel`.

Counts distinguish metadata rows, unique tissues, and image stores. Different resolutions of the same tissue are not independent samples. Missing cell tables mean **unavailable**, not zero cells.

五个数据集里哪些细胞类型能进训练集，写在独立笔记本 `spora_celltype_training_screen.ipynb`。中文说明：`spora_CellData.md`。


In [ ]:
from pathlib import Path
import json
import importlib.metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import zarr
assert int(zarr.__version__.split(".")[0]) >= 3, "OME-Zarr v3 images require zarr>=3 (Python 3.11+)"
from IPython.display import display

DATA_ROOT = Path('/home/lingyu/ssd2/Python/Hist2Pheno/data/CODEX/spora')
DATASETS = ['haist2026lymph', 'hickey2023organization', 'hoefflin2026spatial']
ROOTS = {name: DATA_ROOT / name / name for name in DATASETS}
assert all(p.is_dir() for p in ROOTS.values()), 'Check DATA_ROOT and nested dataset folders'
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})
print({package: importlib.metadata.version(package) for package in ['numpy', 'pandas', 'pyarrow', 'matplotlib', 'zarr']})

## 1. Load metadata and inspect schemas
Tissue metadata may contain one row per modality. Channel availability tables use tissue IDs as their index; their coverage may differ from tissue metadata or local images.

In [ ]:
tables = {}
for name, root in ROOTS.items():
    tables[name] = {
        'tissues': pd.read_parquet(root / 'metadata/tissues.parquet'),
        'channels': pd.read_parquet(root / 'codex/channels.parquet'),
        'availability': pd.read_parquet(root / 'codex/channels_per_tissue.parquet'),
    }
    print(f'\n{name}')
    for kind, frame in tables[name].items():
        print(f'{kind}: {frame.shape}; index={frame.index.name}')
        display(frame.head(3))

## 2. Dataset overview and metadata quality
Patients and tissues are counted uniquely within each study. Paired tissues have both `codex` and `he` in metadata; local file pairing is checked separately below. Clinical fields repeated across tissues should not be treated as independent patient observations.

In [ ]:
summary_rows, quality_rows = [], []
for name, data in tables.items():
    t, ch, av = data['tissues'], data['channels'], data['availability']
    modalities = t.groupby('tissue_id')['modality'].agg(set)
    summary_rows.append(dict(
        dataset=name, metadata_rows=len(t), patients=t.patient_id.nunique(),
        tissues=t.tissue_id.nunique(), codex_tissues=t.loc[t.modality.eq('codex'), 'tissue_id'].nunique(),
        he_tissues=t.loc[t.modality.eq('he'), 'tissue_id'].nunique(),
        paired_tissues=sum({'codex', 'he'}.issubset(x) for x in modalities),
        panel_channels=len(ch), qc_pass_channels=int(ch.qc_pass.fillna(False).sum()),
        nuclear_channels=int(ch.is_nuclear_marker.fillna(False).sum()),
        channel_table_tissues=len(av), median_available_channels=av.fillna(False).sum(axis=1).median(),
        cell_annotations_available=(ROOTS[name] / 'metadata/cells.parquet').exists()))
    quality_rows.append(dict(dataset=name,
        duplicate_tissue_modality_rows=int(t.duplicated(['tissue_id', 'modality']).sum()),
        missing_tissue_ids=int(t.tissue_id.isna().sum()), missing_patient_ids=int(t.patient_id.isna().sum()),
        tissues_with_multiple_splits=int((t.groupby('tissue_id').split.nunique() > 1).sum()),
        patients_with_multiple_splits=int((t.groupby('patient_id').split.nunique() > 1).sum()),
        duplicated_channel_names=int(ch.channel_name.duplicated().sum()),
        duplicate_channel_table_tissue_ids=int(av.index.duplicated().sum())))
summary = pd.DataFrame(summary_rows).set_index('dataset')
summary = summary[['cell_annotations_available'] + [col for col in summary.columns if col != 'cell_annotations_available']]
display(summary)
display(pd.DataFrame(quality_rows).set_index('dataset'))
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, cols, title in zip(axes,
    [['patients', 'tissues'], ['codex_tissues', 'he_tissues', 'paired_tissues'], ['panel_channels', 'qc_pass_channels']],
    ['Study size', 'Metadata modality coverage', 'CODEX channel panel']):
    summary[cols].plot.bar(ax=ax, rot=20)
    ax.set(title=title, xlabel='', ylabel='Count')
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, data) in zip(axes, tables.items()):
    t = data['tissues']
    tissue_split = t.drop_duplicates('tissue_id')['split'].fillna('(missing)').value_counts()
    tissue_split.plot.bar(ax=ax, color='steelblue', rot=30)
    ax.set(title=name, xlabel='Split', ylabel='Unique tissues')
    print(f'\n{name}: missing metadata (%)')
    display(t.isna().mean().mul(100).sort_values(ascending=False).rename('missing_percent').to_frame())
plt.tight_layout()
plt.show()

## 3. Channel availability and panel overlap
Coverage is the fraction of rows marked True in `channels_per_tissue`, with missing values treated as unavailable. Panel overlap uses **exact channel names**; spelling, case, and aliases are not harmonized, so overlap is conservative and does not imply comparable intensities.

In [ ]:
coverage = pd.DataFrame({name: data['availability'].fillna(False).mean() for name, data in tables.items()})
fig, ax = plt.subplots(figsize=(8, max(8, len(coverage) * 0.15)))
image = ax.imshow(np.ma.masked_invalid(coverage.to_numpy(float)), aspect='auto', vmin=0, vmax=1, cmap='viridis')
ax.set_xticks(range(len(DATASETS)), DATASETS, rotation=20, ha='right')
ax.set_yticks(range(len(coverage)), coverage.index, fontsize=7)
ax.set_title('Channel availability by study (grey = absent from panel)')
ax.set_facecolor('lightgrey')
fig.colorbar(image, ax=ax, label='Fraction of channel-table tissues')
plt.tight_layout()
plt.show()
panels = {name: set(data['channels'].channel_name) for name, data in tables.items()}
intersection = pd.DataFrame({a: {b: len(panels[a] & panels[b]) for b in DATASETS} for a in DATASETS})
display(intersection.rename_axis('Exact-name shared channel count'))
print('Shared across all three:', sorted(set.intersection(*panels.values())))
fig, ax = plt.subplots(figsize=(8, 4))
ax.boxplot([data['availability'].fillna(False).sum(axis=1).to_numpy() for data in tables.values()], tick_labels=DATASETS)
ax.tick_params(axis='x', rotation=20)
ax.set(ylabel='Available channels per tissue', title='Per-tissue panel size')
plt.tight_layout()
plt.show()

## 4. Local image inventory and dimensions
List OME-Zarr stores at each downloaded resolution without scanning image chunks. Dimension statistics use only **1 µm/pixel** images. Image area is the rectangular field of view, not segmented tissue area. Uncompressed sizes are theoretical array sizes, not disk usage.

In [ ]:
store_rows, image_rows = [], []
for name, root in ROOTS.items():
    for modality in ['codex', 'he']:
        for folder in sorted((root / modality).glob('*/images')):
            stores = sorted(folder.glob('*.ome.zarr'))
            store_rows.append(dict(dataset=name, modality=modality, resolution=folder.parent.name, stores=len(stores)))
            if folder.parent.name != '1_0mpp':
                continue
            for store in stores:
                meta = json.loads((store / 'zarr.json').read_text())['attributes']['ome']
                ms = meta['multiscales'][0]
                level = ms['datasets'][0]
                array_meta = json.loads((store / level['path'] / 'zarr.json').read_text())
                dims = dict(zip([a['name'] for a in ms['axes']], array_meta['shape']))
                scale = next(v['scale'] for v in level['coordinateTransformations'] if v['type'] == 'scale')
                scales = dict(zip([a['name'] for a in ms['axes']], scale))
                image_rows.append(dict(dataset=name, modality=modality, tissue_id=store.name.removesuffix('.ome.zarr'),
                    channels=dims.get('c', 1), height=dims['y'], width=dims['x'], dtype=array_meta['data_type'],
                    field_area_mm2=dims['y']*scales['y']*dims['x']*scales['x']/1e6,
                    uncompressed_GiB=int(np.prod(array_meta['shape']))*np.dtype(array_meta['data_type']).itemsize/2**30,
                    path=str(store)))
store_inventory = pd.DataFrame(store_rows)
images = pd.DataFrame(image_rows)
display(store_inventory)
display(images.groupby(['dataset', 'modality'])[['height', 'width', 'channels', 'field_area_mm2', 'uncompressed_GiB']].agg(['min', 'median', 'max']))
for name, data in tables.items():
    local = images.loc[images.dataset.eq(name)]
    codex_ids = set(local.loc[local.modality.eq('codex'), 'tissue_id'])
    he_ids = set(local.loc[local.modality.eq('he'), 'tissue_id'])
    metadata_ids = set(data['tissues'].loc[data['tissues'].modality.eq('codex'), 'tissue_id'])
    availability_ids = set(data['availability'].index)
    print(f'\n{name}: local paired stores={len(codex_ids & he_ids)}')
    for label, ids in [('CODEX metadata without local store', metadata_ids-codex_ids),
                       ('Local CODEX store without metadata', codex_ids-metadata_ids),
                       ('Channel-table IDs without CODEX metadata', availability_ids-metadata_ids),
                       ('CODEX metadata without channel-table row', metadata_ids-availability_ids)]:
        print(f'  {label}: {len(ids)}; examples={sorted(map(str, ids))[:5]}')
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name in DATASETS:
    subset = images.loc[images.dataset.eq(name) & images.modality.eq('codex')]
    axes[0].scatter(subset.width, subset.height, label=name, alpha=0.6, s=15)
    axes[1].hist(subset.field_area_mm2, bins=20, alpha=0.5, label=name)
axes[0].set(xlabel='Width (pixels)', ylabel='Height (pixels)', title='CODEX dimensions at 1 µm/pixel')
axes[1].set(xlabel='Rectangular field area (mm²)', ylabel='Image count', title='CODEX field area')
for ax in axes: ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 5. Representative CODEX image crops
Show a central crop of at most 512 × 512 pixels from the first sorted CODEX image in each study at 1 µm/pixel. Labels come from the image itself, preserving channel order. Each channel is independently stretched to its 1st–99th percentile; brightness is **not comparable** across panels or studies. Central crops can contain background and are not representative sampling of tissue biology.

In [ ]:
CROP_SIZE = 512
preview_stats = []
fig, axes = plt.subplots(3, 4, figsize=(13, 10), squeeze=False)
for row, name in enumerate(DATASETS):
    path = Path(images.loc[images.dataset.eq(name) & images.modality.eq('codex')].sort_values('tissue_id').iloc[0].path)
    group = zarr.open_group(str(path), mode='r')
    ome = group.attrs['ome']
    ms = ome['multiscales'][0]
    assert [a['name'] for a in ms['axes']] == ['c', 'y', 'x'], 'Expected CYX image axes'
    array = group[ms['datasets'][0]['path']]
    labels = [ch['label'] for ch in ome['omero']['channels']]
    chosen = []
    for alternatives in [('DAPI', 'DRAQ5', 'Hoechst2', 'HOECHST2'), ('CD3',), ('CD8',), ('CD68', 'CD45')]:
        idx = next((labels.index(label) for label in alternatives if label in labels and labels.index(label) not in chosen), None)
        if idx is None: idx = next(i for i in range(len(labels)) if i not in chosen)
        chosen.append(idx)
    y0, x0 = max(0, (array.shape[1]-CROP_SIZE)//2), max(0, (array.shape[2]-CROP_SIZE)//2)
    for col, idx in enumerate(chosen):
        crop = np.asarray(array[idx, y0:y0+CROP_SIZE, x0:x0+CROP_SIZE])
        lo, hi = np.percentile(crop, [1, 99])
        axes[row, col].imshow(crop, cmap='gray', vmin=lo, vmax=max(hi, lo+1))
        axes[row, col].set_title(f'{name}\n{labels[idx]}', fontsize=9)
        axes[row, col].axis('off')
        preview_stats.append(dict(dataset=name, tissue_id=path.name.removesuffix('.ome.zarr'), channel=labels[idx],
            crop_y=y0, crop_x=x0, mean=float(crop.mean()), std=float(crop.std()),
            p01=lo, median=float(np.median(crop)), p99=hi, zero_fraction=float((crop == 0).mean())))
plt.tight_layout()
plt.show()
display(pd.DataFrame(preview_stats))

## 6. Available cell annotations
Only studies with `metadata/cells.parquet` contribute here. Counts describe the supplied annotations; no segmentation or cell calling is performed. The cell table contains tissue IDs and labels but no coordinate columns. Separate instance masks provide spatial locations for the H&E overlay at the end of this notebook. Tissue-level cell counts are not patient-level replicates.

In [ ]:
cell_summaries = []
for name, root in ROOTS.items():
    path = root / 'metadata/cells.parquet'
    if not path.exists():
        print(f'{name}: cell annotations unavailable')
        continue
    cells = pd.read_parquet(path)
    counts = cells.groupby('tissue_id').size().rename('cells').sort_values(ascending=False)
    cell_summaries.append(dict(dataset=name, annotated_cells=len(cells), annotated_tissues=len(counts),
        median_cells_per_tissue=counts.median(), duplicate_tissue_cell_ids=int(cells.duplicated(['tissue_id', 'cell_id']).sum()),
        tissues_absent_from_metadata=len(set(counts.index)-set(tables[name]['tissues'].tissue_id))))
    display(cells.isna().mean().mul(100).rename('missing_percent').to_frame())
    display(counts.describe().to_frame())
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    counts.plot.bar(ax=axes[0], color='steelblue')
    axes[0].set(title=f'{name}: cells per tissue', xlabel='Tissue ID', ylabel='Annotated cells')
    axes[0].tick_params(axis='x', labelsize=6)
    for ax, field in zip(axes[1:], ['cell_type', 'cell_type_atlas']):
        labels = cells[field].fillna('(missing)').value_counts()
        display(labels.rename('cells').to_frame().assign(percent=lambda x: 100*x.cells/len(cells)))
        labels.head(20).sort_values().plot.barh(ax=ax)
        ax.set(title=f'{field} (top 20)', xlabel='Annotated cells', ylabel='')
    plt.tight_layout()
    plt.show()
    composition = pd.crosstab(cells.tissue_id, cells.cell_type_atlas.fillna('(missing)'), normalize='index')
    composition.plot.bar(stacked=True, figsize=(13, 5), width=0.9, colormap='tab20')
    plt.ylabel('Fraction of annotated cells')
    plt.title(f'{name}: atlas cell composition by tissue')
    plt.xlabel('Tissue ID')
    plt.xticks(fontsize=6)
    plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
    plt.tight_layout()
    plt.show()
    del cells
if cell_summaries:
    display(pd.DataFrame(cell_summaries).set_index('dataset'))

## Interpretation notes
- Use local image inventory alongside metadata counts: the supplied tables and downloaded images need not cover identical tissue sets.
- Channel presence and `qc_pass` are distinct: presence describes the tissue panel, while QC flags come from the study channel table.
- Exact channel-name overlap does not resolve marker aliases or normalize assay measurements.
- Crop intensity summaries include background and describe only the displayed crop. They are not whole-slide expression estimates.
- All analysis is descriptive; no between-study biological comparison or statistical significance is inferred.

### See haist2026lymph

Dataset-specific descriptive results, using only **haist2026lymph**. This section can run independently of the multi-study analysis above. Tissues and patients are counted uniquely; clinical metadata repeated across modalities are deduplicated. Image dimensions use 1 µm/pixel stores, and previews use bounded central crops with per-channel contrast stretching. Rectangular field area is not segmented tissue area.


In [ ]:
from pathlib import Path
import json
import importlib.metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import zarr
assert int(zarr.__version__.split(".")[0]) >= 3, "OME-Zarr v3 haist_images require zarr>=3 (Python 3.11+)"
from IPython.display import display

DATA_ROOT = Path('/home/lingyu/ssd2/Python/Hist2Pheno/data/CODEX/spora')
haist_DATASETS = ['haist2026lymph']
haist_ROOTS = {name: DATA_ROOT / name / name for name in haist_DATASETS}
assert all(p.is_dir() for p in haist_ROOTS.values()), 'Check DATA_ROOT and nested dataset folders'
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})
print({package: importlib.metadata.version(package) for package in ['numpy', 'pandas', 'pyarrow', 'matplotlib', 'zarr']})
haist_tables = {}
for name, root in haist_ROOTS.items():
    haist_tables[name] = {
        'tissues': pd.read_parquet(root / 'metadata/tissues.parquet'),
        'channels': pd.read_parquet(root / 'codex/channels.parquet'),
        'availability': pd.read_parquet(root / 'codex/channels_per_tissue.parquet'),
    }
    print(f'\n{name}')
    for kind, frame in haist_tables[name].items():
        print(f'{kind}: {frame.shape}; index={frame.index.name}')
        display(frame.head(3))

#### Cohort, modality pairing, and metadata quality


In [ ]:
haist_summary_rows, haist_quality_rows = [], []
for name, data in haist_tables.items():
    t, ch, av = data['tissues'], data['channels'], data['availability']
    modalities = t.groupby('tissue_id')['modality'].agg(set)
    haist_summary_rows.append(dict(
        dataset=name, metadata_rows=len(t), patients=t.patient_id.nunique(),
        tissues=t.tissue_id.nunique(), codex_tissues=t.loc[t.modality.eq('codex'), 'tissue_id'].nunique(),
        he_tissues=t.loc[t.modality.eq('he'), 'tissue_id'].nunique(),
        paired_tissues=sum({'codex', 'he'}.issubset(x) for x in modalities),
        panel_channels=len(ch), qc_pass_channels=int(ch.qc_pass.fillna(False).sum()),
        nuclear_channels=int(ch.is_nuclear_marker.fillna(False).sum()),
        channel_table_tissues=len(av), median_available_channels=av.fillna(False).sum(axis=1).median(),
        cell_annotations_available=(haist_ROOTS[name] / 'metadata/cells.parquet').exists()))
    haist_quality_rows.append(dict(dataset=name,
        duplicate_tissue_modality_rows=int(t.duplicated(['tissue_id', 'modality']).sum()),
        missing_tissue_ids=int(t.tissue_id.isna().sum()), missing_patient_ids=int(t.patient_id.isna().sum()),
        tissues_with_multiple_splits=int((t.groupby('tissue_id').split.nunique() > 1).sum()),
        patients_with_multiple_splits=int((t.groupby('patient_id').split.nunique() > 1).sum()),
        duplicated_channel_names=int(ch.channel_name.duplicated().sum()),
        duplicate_channel_table_tissue_ids=int(av.index.duplicated().sum())))
haist_summary = pd.DataFrame(haist_summary_rows).set_index('dataset')
display(haist_summary)
display(pd.DataFrame(haist_quality_rows).set_index('dataset'))
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, cols, title in zip(axes,
    [['patients', 'tissues'], ['codex_tissues', 'he_tissues', 'paired_tissues'], ['panel_channels', 'qc_pass_channels']],
    ['Study size', 'Metadata modality haist_coverage', 'CODEX channel panel']):
    haist_summary[cols].plot.bar(ax=ax, rot=20)
    ax.set(title=title, xlabel='', ylabel='Count')
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()
fig, axes = plt.subplots(1, 1, figsize=(6, 4))
axes = np.atleast_1d(axes)
for ax, (name, data) in zip(axes, haist_tables.items()):
    t = data['tissues']
    tissue_split = t.drop_duplicates('tissue_id')['split'].fillna('(missing)').value_counts()
    tissue_split.plot.bar(ax=ax, color='steelblue', rot=30)
    ax.set(title=name, xlabel='Split', ylabel='Unique tissues')
    print(f'\n{name}: missing metadata (%)')
    display(t.isna().mean().mul(100).sort_values(ascending=False).rename('missing_percent').to_frame())
plt.tight_layout()
plt.show()

In [ ]:
haist_tissues = haist_tables['haist2026lymph']['tissues']
haist_unique_tissues = haist_tissues.drop_duplicates('tissue_id')
print('Unique tissues per patient:')
display(haist_unique_tissues.groupby('patient_id').size().rename('tissues').describe().to_frame())
print('Patient counts per split (patients can appear in multiple splits; see quality checks):')
display(haist_tissues[['patient_id', 'split']].drop_duplicates().groupby('split', dropna=False).patient_id.nunique().rename('patients').to_frame())
print('Metadata values per field (unique tissue rows; missing values included):')
for haist_field in haist_unique_tissues.columns:
    if haist_field not in ['tissue_id', 'patient_id', 'modality'] and haist_unique_tissues[haist_field].nunique() <= 30:
        display(haist_unique_tissues[haist_field].value_counts(dropna=False).rename('tissues').to_frame().rename_axis(haist_field))


#### Channel panel and tissue coverage
Presence and panel QC are reported separately. Missing availability values count as unavailable.


In [ ]:
haist_channels = haist_tables['haist2026lymph']['channels'].copy()
haist_availability = haist_tables['haist2026lymph']['availability'].fillna(False)
haist_channel_coverage = haist_availability.mean().rename('available_fraction')
display(haist_channels.merge(haist_channel_coverage, left_on='channel_name', right_index=True, how='outer'))
fig, axes = plt.subplots(1, 2, figsize=(15, max(5, len(haist_channel_coverage)*0.18)))
haist_channel_coverage.sort_values().plot.barh(ax=axes[0])
axes[0].set(xlabel='Fraction of channel-table tissues', ylabel='Channel', title='haist2026lymph: channel coverage', xlim=(0, 1.05))
haist_availability.sum(axis=1).value_counts().sort_index().plot.bar(ax=axes[1])
axes[1].set(xlabel='Available channels', ylabel='Tissues', title='Available panel size per tissue')
plt.tight_layout()
plt.show()


#### Local image inventory, pairing, and dimensions


In [ ]:
haist_store_rows, haist_image_rows = [], []
for name, root in haist_ROOTS.items():
    for modality in ['codex', 'he']:
        for folder in sorted((root / modality).glob('*/images')):
            stores = sorted(folder.glob('*.ome.zarr'))
            haist_store_rows.append(dict(dataset=name, modality=modality, resolution=folder.parent.name, stores=len(stores)))
            if folder.parent.name != '1_0mpp':
                continue
            for store in stores:
                meta = json.loads((store / 'zarr.json').read_text())['attributes']['ome']
                ms = meta['multiscales'][0]
                level = ms['datasets'][0]
                array_meta = json.loads((store / level['path'] / 'zarr.json').read_text())
                dims = dict(zip([a['name'] for a in ms['axes']], array_meta['shape']))
                scale = next(v['scale'] for v in level['coordinateTransformations'] if v['type'] == 'scale')
                scales = dict(zip([a['name'] for a in ms['axes']], scale))
                haist_image_rows.append(dict(dataset=name, modality=modality, tissue_id=store.name.removesuffix('.ome.zarr'),
                    channels=dims.get('c', 1), height=dims['y'], width=dims['x'], dtype=array_meta['data_type'],
                    field_area_mm2=dims['y']*scales['y']*dims['x']*scales['x']/1e6,
                    uncompressed_GiB=int(np.prod(array_meta['shape']))*np.dtype(array_meta['data_type']).itemsize/2**30,
                    path=str(store)))
haist_store_inventory = pd.DataFrame(haist_store_rows)
haist_images = pd.DataFrame(haist_image_rows)
display(haist_store_inventory)
display(haist_images.groupby(['dataset', 'modality'])[['height', 'width', 'channels', 'field_area_mm2', 'uncompressed_GiB']].agg(['min', 'median', 'max']))
for name, data in haist_tables.items():
    local = haist_images.loc[haist_images.dataset.eq(name)]
    codex_ids = set(local.loc[local.modality.eq('codex'), 'tissue_id'])
    he_ids = set(local.loc[local.modality.eq('he'), 'tissue_id'])
    metadata_ids = set(data['tissues'].loc[data['tissues'].modality.eq('codex'), 'tissue_id'])
    availability_ids = set(data['availability'].index)
    print(f'\n{name}: local paired stores={len(codex_ids & he_ids)}')
    for label, ids in [('CODEX metadata without local store', metadata_ids-codex_ids),
                       ('Local CODEX store without metadata', codex_ids-metadata_ids),
                       ('Channel-table IDs without CODEX metadata', availability_ids-metadata_ids),
                       ('CODEX metadata without channel-table row', metadata_ids-availability_ids)]:
        print(f'  {label}: {len(ids)}; examples={sorted(map(str, ids))[:5]}')
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name in haist_DATASETS:
    subset = haist_images.loc[haist_images.dataset.eq(name) & haist_images.modality.eq('codex')]
    axes[0].scatter(subset.width, subset.height, label=name, alpha=0.6, s=15)
    axes[1].hist(subset.field_area_mm2, bins=20, alpha=0.5, label=name)
axes[0].set(xlabel='Width (pixels)', ylabel='Height (pixels)', title='CODEX dimensions at 1 µm/pixel')
axes[1].set(xlabel='Rectangular field area (mm²)', ylabel='Image count', title='CODEX field area')
for ax in axes: ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

#### Representative CODEX crop
The first sorted tissue is shown; this central crop is not a representative biological sample. Intensities are stretched independently to the 1st–99th percentiles and are not quantitative expression comparisons.


In [ ]:
haist_CROP_SIZE = 512
haist_preview_stats = []
fig, axes = plt.subplots(1, 4, figsize=(13, 4), squeeze=False)
for row, name in enumerate(haist_DATASETS):
    path = Path(haist_images.loc[haist_images.dataset.eq(name) & haist_images.modality.eq('codex')].sort_values('tissue_id').iloc[0].path)
    group = zarr.open_group(str(path), mode='r')
    ome = group.attrs['ome']
    ms = ome['multiscales'][0]
    assert [a['name'] for a in ms['axes']] == ['c', 'y', 'x'], 'Expected CYX image axes'
    array = group[ms['datasets'][0]['path']]
    labels = [ch['label'] for ch in ome['omero']['channels']]
    chosen = []
    for alternatives in [('DAPI', 'DRAQ5', 'Hoechst2', 'HOECHST2'), ('CD3',), ('CD8',), ('CD68', 'CD45')]:
        idx = next((labels.index(label) for label in alternatives if label in labels and labels.index(label) not in chosen), None)
        if idx is None: idx = next(i for i in range(len(labels)) if i not in chosen)
        chosen.append(idx)
    y0, x0 = max(0, (array.shape[1]-haist_CROP_SIZE)//2), max(0, (array.shape[2]-haist_CROP_SIZE)//2)
    for col, idx in enumerate(chosen):
        crop = np.asarray(array[idx, y0:y0+haist_CROP_SIZE, x0:x0+haist_CROP_SIZE])
        lo, hi = np.percentile(crop, [1, 99])
        axes[row, col].imshow(crop, cmap='gray', vmin=lo, vmax=max(hi, lo+1))
        axes[row, col].set_title(f'{name}\n{labels[idx]}', fontsize=9)
        axes[row, col].axis('off')
        haist_preview_stats.append(dict(dataset=name, tissue_id=path.name.removesuffix('.ome.zarr'), channel=labels[idx],
            crop_y=y0, crop_x=x0, mean=float(crop.mean()), std=float(crop.std()),
            p01=lo, median=float(np.median(crop)), p99=hi, zero_fraction=float((crop == 0).mean())))
plt.tight_layout()
plt.show()
display(pd.DataFrame(haist_preview_stats))

#### Cell annotation availability
Missing annotation files indicate unavailable annotations, not zero cells. No segmentation or cell calling is performed.


In [ ]:
haist_cell_summaries = []
for name, root in haist_ROOTS.items():
    path = root / 'metadata/cells.parquet'
    if not path.exists():
        print(f'{name}: cell annotations unavailable')
        continue
    cells = pd.read_parquet(path)
    counts = cells.groupby('tissue_id').size().rename('cells').sort_values(ascending=False)
    haist_cell_summaries.append(dict(dataset=name, annotated_cells=len(cells), annotated_tissues=len(counts),
        median_cells_per_tissue=counts.median(), duplicate_tissue_cell_ids=int(cells.duplicated(['tissue_id', 'cell_id']).sum()),
        tissues_absent_from_metadata=len(set(counts.index)-set(haist_tables[name]['tissues'].tissue_id))))
    display(cells.isna().mean().mul(100).rename('missing_percent').to_frame())
    display(counts.describe().to_frame())
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    counts.plot.bar(ax=axes[0], color='steelblue')
    axes[0].set(title=f'{name}: cells per tissue', xlabel='Tissue ID', ylabel='Annotated cells')
    axes[0].tick_params(axis='x', labelsize=6)
    for ax, field in zip(axes[1:], ['cell_type', 'cell_type_atlas']):
        labels = cells[field].fillna('(missing)').value_counts()
        display(labels.rename('cells').to_frame().assign(percent=lambda x: 100*x.cells/len(cells)))
        labels.head(20).sort_values().plot.barh(ax=ax)
        ax.set(title=f'{field} (top 20)', xlabel='Annotated cells', ylabel='')
    plt.tight_layout()
    plt.show()
    composition = pd.crosstab(cells.tissue_id, cells.cell_type_atlas.fillna('(missing)'), normalize='index')
    composition.plot.bar(stacked=True, figsize=(13, 5), width=0.9, colormap='tab20')
    plt.ylabel('Fraction of annotated cells')
    plt.title(f'{name}: atlas cell composition by tissue')
    plt.xlabel('Tissue ID')
    plt.xticks(fontsize=6)
    plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
    plt.tight_layout()
    plt.show()
    del cells
if haist_cell_summaries:
    display(pd.DataFrame(haist_cell_summaries).set_index('dataset'))